In [ ]:
# Auto-install PyWavelets for Run-All robustness
import importlib
import subprocess
import sys

if importlib.util.find_spec('pywt') is None:
    print('Installing PyWavelets...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'PyWavelets', '-q'])
else:
    print('PyWavelets already installed.')

In [ ]:
import random
import shutil
from collections import Counter, defaultdict
from pathlib import Path

import albumentations as A
import cv2
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO

try:
    import pywt
    _HAS_PYWT = True
except Exception:
    pywt = None
    _HAS_PYWT = False

# ── Config ─────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

DATASET_ROOT = Path(DATASET_ROOT) if 'DATASET_ROOT' in globals() else Path.cwd()
CATEGORIES = CATEGORIES if 'CATEGORIES' in globals() else [p.name for p in DATASET_ROOT.iterdir() if p.is_dir() and (p / 'test').exists()]
IMG_SIZE = IMG_SIZE if 'IMG_SIZE' in globals() else 800
# PyTorch CUDA — ne pas utiliser cv2.cuda (OpenCV GPU ≠ entraînement YOLO)
DEVICE = DEVICE if 'DEVICE' in globals() else ('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device utilisé pour train YOLO: {DEVICE} | cuda disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU PyTorch: {torch.cuda.get_device_name(0)}')

YOLO_ROOT_ENH = DATASET_ROOT / 'yolo_multiclass_dataset_good_wavelet'
RUN_NAME_ENH  = 'mvtec3d_good_wavelet'

# Keep your previous base model choice
BASE_MODEL_ENH = BASE_MODEL if 'BASE_MODEL' in globals() else 'yolo26m-seg.pt'
EPOCHS_ENH     = 100
BATCH_ENH      = 8

# Wavelet knobs
USE_WAVELET = True and _HAS_PYWT
WAVELET_NAME = 'db2'
WAVELET_BOOST = 1.6  # >1 boosts high-frequency details

# Imbalance knobs (no class-weights in YOLO)
MIN_SAMPLES_PER_CLASS = 280
MAX_MULTIPLIER_PER_CLASS = 22  # aug copies ≈ (mult-1)*n per rare defect class

# 'good' downsampling — évite que la majorité écrase les défauts
GOOD_CAP_MULT_VS_DEFECT_MEDIAN_TRAIN = 1.15  # train good ≤ max(MIN_TRAIN, médiane(defauts)*this)
GOOD_CAP_MULT_VS_DEFECT_MEDIAN_VAL = 1.20
GOOD_CAP_MULT_VS_DEFECT_MEDIAN_TEST = 1.20
MIN_GOOD_KEEP_TRAIN = 80
MIN_GOOD_KEEP_VAL = 24
MIN_GOOD_KEEP_TEST = 24
# Après calcul du target sur les défauts, on peut encore réduire good
GOOD_POST_BALANCE_MULT = 1.08  # good train ≤ target * this


def _list_pngs(folder: Path):
    return sorted(folder.glob('*.png')) if folder.exists() else []


def build_class_map_with_good(dataset_root: Path, categories):
    """Real defect types only — 'combined' is not a class (multi-defect in one GT)."""
    defects = set()
    for cat in categories:
        test_root = dataset_root / cat / 'test'
        if not test_root.exists():
            continue
        for d in test_root.iterdir():
            if d.is_dir() and d.name not in ('good', 'combined'):
                defects.add(d.name)

    class_names = ['good'] + sorted(defects)
    class_map = {name: i for i, name in enumerate(class_names)}
    return class_names, class_map


CLASS_NAMES_ENH, CLASS_MAP_ENH = build_class_map_with_good(DATASET_ROOT, CATEGORIES)
# Per-category pixel value -> defect (for decoding MVTec 3D-AD 'combined' masks)
CAT_PIXEL_MAP_ENH = {}
for cat in CATEGORIES:
    test_root = DATASET_ROOT / cat / 'test'
    if not test_root.exists():
        continue
    non_combined = sorted([
        d.name for d in test_root.iterdir()
        if d.is_dir() and d.name not in ('good', 'combined')
    ])
    CAT_PIXEL_MAP_ENH[cat] = {255 - i: defect for i, defect in enumerate(non_combined)}

print('CAT_PIXEL_MAP_ENH (for combined GT decode):')
for cat, pmap in sorted(CAT_PIXEL_MAP_ENH.items()):
    print(f'  {cat:<12} {pmap}')

NC_ENH = len(CLASS_NAMES_ENH)
print('Enhanced classes:', CLASS_NAMES_ENH)
print('Wavelet enabled :', USE_WAVELET, f'(pywt available: {_HAS_PYWT})')


def _sample_record(rgb_path: Path, gt_path: Path, defect_name: str, cat: str, gt_mask=None):
    """gt_mask: optional uint8 binary mask for 'combined' expanded samples."""
    rec = {
        'rgb': rgb_path,
        'gt': gt_path,
        'defect': defect_name,
        'cat': cat,
        'class_id': CLASS_MAP_ENH[defect_name],
    }
    if gt_mask is not None:
        rec['gt_mask'] = gt_mask
    return rec


def _append_defect_split(rgb_path: Path, gt_dir: Path, cat: str, folder_defect: str,
                         out_list: list, is_combined: bool):
    """Add one or more samples to out_list for this RGB file."""
    gt_f = gt_dir / rgb_path.name
    if not gt_f.exists():
        return
    gt_arr = np.array(Image.open(gt_f))
    if gt_arr.max() == 0:
        return
    if is_combined:
        for pval, defect_name in CAT_PIXEL_MAP_ENH.get(cat, {}).items():
            if defect_name not in CLASS_MAP_ENH:
                continue
            binary_mask = (gt_arr == pval).astype(np.uint8) * 255
            if binary_mask.max() == 0:
                continue
            out_list.append(_sample_record(rgb_path, gt_f, defect_name, cat, gt_mask=binary_mask))
    else:
        out_list.append(_sample_record(rgb_path, gt_f, folder_defect, cat))


def downsample_good_vs_defects(samples: list, min_keep: int, cap_mult: float) -> list:
    """Réduit les `good` pour ne pas excéder ~cap_mult * médiane(effectifs défauts)."""
    good_id = CLASS_MAP_ENH['good']
    good = [s for s in samples if s['class_id'] == good_id]
    rest = [s for s in samples if s['class_id'] != good_id]
    if not good or not rest:
        return samples
    dc = [c for cid, c in Counter(s['class_id'] for s in rest).items() if cid != good_id]
    if not dc:
        return samples
    med = float(np.median(dc))
    cap = max(min_keep, int(med * cap_mult))
    if len(good) <= cap:
        return samples
    random.shuffle(good)
    return rest + good[:cap]


def collect_samples_with_good(dataset_root: Path, categories, val_ratio=0.2):
    """
    Like the main notebook: 'combined' is not a class.
    Combined folder images are expanded — one sample per real defect mask (gt_mask + class_id).
    """
    train_samples, val_samples, test_samples = [], [], []

    for cat in categories:
        cat_root = dataset_root / cat

        # GOOD from train -> split train/val
        good_train_dir = cat_root / 'train' / 'good' / 'rgb'
        good_train_imgs = _list_pngs(good_train_dir)
        random.shuffle(good_train_imgs)
        n_val_good = int(len(good_train_imgs) * val_ratio)
        val_good = good_train_imgs[:n_val_good]
        train_good = good_train_imgs[n_val_good:]

        train_samples += [_sample_record(p, None, 'good', cat) for p in train_good]
        val_samples += [_sample_record(p, None, 'good', cat) for p in val_good]

        # GOOD from test -> test
        good_test_dir = cat_root / 'test' / 'good' / 'rgb'
        test_samples += [_sample_record(p, None, 'good', cat) for p in _list_pngs(good_test_dir)]

        # Defects from test (rgb + gt)
        test_root = cat_root / 'test'
        if not test_root.exists():
            continue

        for defect_dir in sorted(test_root.iterdir()):
            if not defect_dir.is_dir() or defect_dir.name == 'good':
                continue

            defect = defect_dir.name
            rgb_dir = defect_dir / 'rgb'
            gt_dir = defect_dir / 'gt'
            rgb_files = _list_pngs(rgb_dir)
            random.shuffle(rgb_files)

            n_val = int(len(rgb_files) * val_ratio)
            val_rgb = rgb_files[:n_val]
            test_rgb = rgb_files[n_val:]
            is_combined = defect == 'combined'

            for p in val_rgb:
                _append_defect_split(p, gt_dir, cat, defect, val_samples, is_combined)

            for p in test_rgb:
                _append_defect_split(p, gt_dir, cat, defect, test_samples, is_combined)

            for p in rgb_files:
                _append_defect_split(p, gt_dir, cat, defect, train_samples, is_combined)

    train_samples = downsample_good_vs_defects(
        train_samples, MIN_GOOD_KEEP_TRAIN, GOOD_CAP_MULT_VS_DEFECT_MEDIAN_TRAIN)
    val_samples = downsample_good_vs_defects(
        val_samples, MIN_GOOD_KEEP_VAL, GOOD_CAP_MULT_VS_DEFECT_MEDIAN_VAL)
    test_samples = downsample_good_vs_defects(
        test_samples, MIN_GOOD_KEEP_TEST, GOOD_CAP_MULT_VS_DEFECT_MEDIAN_TEST)

    return train_samples, val_samples, test_samples


def wavelet_enhance_rgb(rgb: np.ndarray, wavelet='db2', boost=1.6):
    out = np.zeros_like(rgb)
    for c in range(3):
        ch = rgb[:, :, c].astype(np.float32)
        cA, (cH, cV, cD) = pywt.dwt2(ch, wavelet)
        cH *= boost
        cV *= boost
        cD *= boost
        rec = pywt.idwt2((cA, (cH, cV, cD)), wavelet)
        rec = rec[: ch.shape[0], : ch.shape[1]]
        out[:, :, c] = np.clip(rec, 0, 255).astype(np.uint8)
    return out


def mask_to_yolo_polygons_local(mask: np.ndarray, class_id: int, eps=1.0, min_points=6):
    if mask.ndim == 3:
        mask = mask[:, :, 0]
    H, W = mask.shape[:2]
    bin_mask = (mask > 0).astype(np.uint8) * 255

    contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    lines = []

    for cnt in contours:
        cnt = cv2.approxPolyDP(cnt, epsilon=eps, closed=True)
        if len(cnt) < 3:
            continue
        coords = cnt[:, 0, :].astype(float)
        coords[:, 0] /= W
        coords[:, 1] /= H
        coords = np.clip(coords, 0.0, 1.0)

        flat = coords.reshape(-1)
        if flat.size < min_points:
            continue
        poly = ' '.join(f'{v:.6f}' for v in flat)
        lines.append(f'{class_id} {poly}')

    return lines


def full_image_polygon_line(class_id: int):
    # Rectangle polygon normalized: TL, TR, BR, BL
    return f'{class_id} 0.0 0.0 1.0 0.0 1.0 1.0 0.0 1.0'


augmenter = A.Compose([
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
        A.MedianBlur(blur_limit=5, p=1.0),
    ], p=0.30),
    A.RandomBrightnessContrast(brightness_limit=0.20, contrast_limit=0.20, p=0.35),
    A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=15, val_shift_limit=10, p=0.30),
    A.ShiftScaleRotate(shift_limit=0.08, scale_limit=0.10, rotate_limit=12,
                       border_mode=cv2.BORDER_REFLECT_101, p=0.40),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
])


def class_distribution(samples):
    c = Counter(s['class_id'] for s in samples)
    return {CLASS_NAMES_ENH[k]: v for k, v in sorted(c.items())}


def build_rebalanced_train_set(train_samples):
    """
    Recale les défauts vers un effectif cible (médiane des classes défaut),
    via sur-échantillonnage + plan d'augmentation. `good` est plafonné à ~target.
    """
    good_id = CLASS_MAP_ENH['good']
    by_class = defaultdict(list)
    for s in train_samples:
        by_class[s['class_id']].append(s)

    defect_counts = {cid: len(lst) for cid, lst in by_class.items() if cid != good_id}
    if defect_counts:
        target = max(MIN_SAMPLES_PER_CLASS, int(np.median(list(defect_counts.values()))))
    else:
        target = MIN_SAMPLES_PER_CLASS

    expanded = []
    aug_plan = []

    for cid in sorted(by_class.keys()):
        samples = by_class[cid][:]
        n = len(samples)

        if cid == good_id:
            cap_g = max(MIN_GOOD_KEEP_TRAIN, int(target * GOOD_POST_BALANCE_MULT))
            if n > cap_g:
                random.shuffle(samples)
                samples = samples[:cap_g]
            expanded.extend(samples)
            continue

        expanded.extend(samples)
        if n >= target:
            continue
        deficit = target - n
        max_extra = n * (MAX_MULTIPLIER_PER_CLASS - 1)
        take = min(deficit, max_extra)
        if take <= 0:
            continue
        for s in random.choices(samples, k=take):
            aug_plan.append(s)

    random.shuffle(expanded)
    random.shuffle(aug_plan)
    return expanded, aug_plan, target


def write_yolo_dataset_with_good_wavelet(train_samples, val_samples, test_samples, yolo_root: Path):
    if yolo_root.exists():
        shutil.rmtree(yolo_root)

    for split_samples, split in [(train_samples, 'train'), (val_samples, 'val'), (test_samples, 'test')]:
        (yolo_root / 'images' / split).mkdir(parents=True, exist_ok=True)
        (yolo_root / 'labels' / split).mkdir(parents=True, exist_ok=True)

        for i, s in enumerate(split_samples):
            img_bgr = cv2.imread(str(s['rgb']))
            if img_bgr is None:
                continue
            img = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

            if USE_WAVELET:
                img = wavelet_enhance_rgb(img, wavelet=WAVELET_NAME, boost=WAVELET_BOOST)

            stem = f"{s['cat']}__{s['defect']}__{Path(s['rgb']).stem}__{i:05d}"
            out_img = yolo_root / 'images' / split / f'{stem}.jpg'
            out_lbl = yolo_root / 'labels' / split / f'{stem}.txt'

            if s['defect'] == 'good':
                lines = [full_image_polygon_line(s['class_id'])]
            else:
                gt = s['gt_mask'] if s.get('gt_mask') is not None else np.array(Image.open(s['gt']))
                lines = mask_to_yolo_polygons_local(gt, s['class_id'])

            # Ensure at least one line; skip if broken anomaly mask
            if not lines:
                continue

            cv2.imwrite(str(out_img), cv2.cvtColor(img, cv2.COLOR_RGB2BGR))
            out_lbl.write_text('\n'.join(lines))


def write_augmented_train_only(aug_plan, yolo_root: Path):
    img_dir = yolo_root / 'images' / 'train'
    lbl_dir = yolo_root / 'labels' / 'train'

    for j, s in enumerate(aug_plan):
        base = cv2.cvtColor(cv2.imread(str(s['rgb'])), cv2.COLOR_BGR2RGB)
        if s['defect'] == 'good':
            mask = np.ones(base.shape[:2], dtype=np.uint8) * 255
        else:
            if s.get('gt_mask') is not None:
                mask = (s['gt_mask'] > 0).astype(np.uint8) * 255
            else:
                mask = (np.array(Image.open(s['gt'])) > 0).astype(np.uint8) * 255

        aug = augmenter(image=base, mask=mask)
        img_aug = aug['image']
        mask_aug = aug['mask']

        if USE_WAVELET:
            img_aug = wavelet_enhance_rgb(img_aug, wavelet=WAVELET_NAME, boost=WAVELET_BOOST)

        stem = f"AUG__{s['cat']}__{s['defect']}__{Path(s['rgb']).stem}__{j:05d}"
        out_img = img_dir / f'{stem}.jpg'
        out_lbl = lbl_dir / f'{stem}.txt'

        if s['defect'] == 'good':
            lines = [full_image_polygon_line(s['class_id'])]
        else:
            lines = mask_to_yolo_polygons_local(mask_aug, s['class_id'])

        if not lines:
            continue

        cv2.imwrite(str(out_img), cv2.cvtColor(img_aug, cv2.COLOR_RGB2BGR))
        out_lbl.write_text('\n'.join(lines))


def write_data_yaml(yolo_root: Path):
    names_yaml = '\n'.join(f'  {i}: {n}' for i, n in enumerate(CLASS_NAMES_ENH))
    txt = f"""path: {yolo_root.as_posix()}
train: images/train
val: images/val
test: images/test
nc: {NC_ENH}
names:
{names_yaml}
"""
    (yolo_root / 'data.yaml').write_text(txt)


# ── Build enhanced dataset ─────────────────────────────────────────────────────
train_samples_raw, val_samples, test_samples = collect_samples_with_good(DATASET_ROOT, CATEGORIES)

print('Train (après réduction good vs médiane défauts):', class_distribution(train_samples_raw))
print('Val   (après réduction good):', class_distribution(val_samples))
print('Test  (après réduction good):', class_distribution(test_samples))

train_samples_base, aug_plan, defect_balance_target = build_rebalanced_train_set(train_samples_raw)
print(f'Cible équilibrage défauts (médiane, min={MIN_SAMPLES_PER_CLASS}): {defect_balance_target}')
print(f"Échantillons réservés à l'augmentation offline: {len(aug_plan)}")
print('Train (après plafond good + plan aug):', class_distribution(train_samples_base))

write_yolo_dataset_with_good_wavelet(train_samples_base, val_samples, test_samples, YOLO_ROOT_ENH)
write_augmented_train_only(aug_plan, YOLO_ROOT_ENH)
write_data_yaml(YOLO_ROOT_ENH)

print('\nEnhanced YOLO dataset written to:', YOLO_ROOT_ENH)
print('Planned augmented samples      :', len(aug_plan))

# ── Quick distribution check from written labels ───────────────────────────────
train_counter = Counter()
for f in (YOLO_ROOT_ENH / 'labels' / 'train').glob('*.txt'):
    for line in f.read_text().splitlines():
        if line.strip():
            train_counter[int(line.split()[0])] += 1

print('\nFinal TRAIN instance distribution:')
for cid in range(NC_ENH):
    print(f'  {cid:2d} | {CLASS_NAMES_ENH[cid]:<20}: {train_counter[cid]}')


# ── Train (no class weights) ───────────────────────────────────────────────────
model_enh = YOLO(BASE_MODEL_ENH)
results_enh = model_enh.train(
    data=str(YOLO_ROOT_ENH / 'data.yaml'),
    task='segment',
    imgsz=IMG_SIZE,
    epochs=EPOCHS_ENH,
    batch=BATCH_ENH,
    device=DEVICE,
    project='runs/segment_enhanced',
    name=RUN_NAME_ENH,
    optimizer='auto',
    lr0=1e-3,
    lrf=0.01,
    mosaic=1.0,
    mixup=0.10,
)

run_dir_enh = Path(results_enh.save_dir)
best_model_enh = YOLO(str(run_dir_enh / 'weights' / 'best.pt'))

print('\nTraining done. Best weights:', run_dir_enh / 'weights' / 'best.pt')
print('Data yaml            :', YOLO_ROOT_ENH / 'data.yaml')

In [ ]:
# ── Restart from best.pt (NaN-safe) ───────────────────────────────────────────
# POURQUOI PAS resume=True : last.pt contient l'état AdamW corrompu (moments NaN).
# Le resume restaure aussi cette corruption → NaN persiste indéfiniment.
#
# CAUSE RACINE : full_image_polygon_line génère des coords exactement à 0.0/1.0.
# La perte DFL calcule log(dist_bord) → log(0) = -inf → NaN dans box/cls/dfl_loss.
#
# SOLUTION : repartir des poids sains (best.pt) avec un optimiseur propre + lr réduit.
# Arrêter le training en cours avant d'exécuter cette cellule.

from ultralytics import YOLO
from pathlib import Path

best_pt   = Path('runs/segment/runs/segment_enhanced/mvtec3d_good_wavelet7/weights/best.pt')
data_yaml = Path('yolo_multiclass_dataset_good_wavelet/data.yaml')

if not best_pt.exists():
    raise FileNotFoundError(f'best.pt introuvable : {best_pt}')
if not data_yaml.exists():
    raise FileNotFoundError(f'data.yaml introuvable : {data_yaml} — relancer les cellules de construction du dataset')

EPOCHS_REMAINING = 85  # 100 total - 15 déjà accomplis

model_enh = YOLO(str(best_pt))  # charge les poids sains, PAS l'état optimiseur
results_enh = model_enh.train(
    data=str(data_yaml),
    task='segment',
    imgsz=800,
    epochs=EPOCHS_REMAINING,
    batch=8,
    device=0,
    project='runs/segment_enhanced',
    name='mvtec3d_good_wavelet_fixed',
    optimizer='AdamW',
    lr0=1e-4,       # 10× plus bas que l'original pour éviter la divergence
    lrf=0.01,
    warmup_epochs=1,
    mosaic=1.0,
    mixup=0.10,
)

run_dir_enh = Path(results_enh.save_dir)
best_model_enh = YOLO(str(run_dir_enh / 'weights' / 'best.pt'))
print('Entraînement terminé.')
print('Meilleurs poids :', run_dir_enh / 'weights' / 'best.pt')

New https://pypi.org/project/ultralytics/8.4.48 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.30  Python-3.10.18 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4050 Laptop GPU, 6140MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=yolo_multiclass_dataset_good_wavelet\data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=85, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.1, mode=train, model=runs\segment\runs\segment_enhanced\mvt

In [ ]:
# Prediction helper when 'good' is an explicit class

def predict_image_with_good_class(image_path, model, class_names, conf_threshold=0.25, iou_threshold=0.45):
    res = model.predict(
        source=str(image_path),
        imgsz=IMG_SIZE,
        conf=conf_threshold,
        iou=iou_threshold,
        device=DEVICE,
        save=False,
        verbose=False,
    )[0]

    vis = cv2.cvtColor(res.plot(), cv2.COLOR_BGR2RGB)

    if res.boxes is None or len(res.boxes) == 0:
        return {'status': 'NORMAL', 'defect_types': [], 'confidences': [], 'visualization': vis}

    cls_ids = res.boxes.cls.cpu().numpy().astype(int).tolist()
    confs = res.boxes.conf.cpu().numpy().tolist()
    names = [class_names[i] for i in cls_ids]

    anomaly = [(n, c) for n, c in zip(names, confs) if n != 'good']
    if len(anomaly) == 0:
        return {'status': 'NORMAL', 'defect_types': [], 'confidences': [], 'visualization': vis}

    return {
        'status': 'ANOMALY',
        'defect_types': [a[0] for a in anomaly],
        'confidences': [a[1] for a in anomaly],
        'visualization': vis,
    }


# Example:
# pred = predict_image_with_good_class('path/to/image.png', best_model_enh, CLASS_NAMES_ENH)
# print(pred['status'], pred['defect_types'])